# Homework 2 - Xinlei Cheng

In [0]:
from pyspark.sql.functions import (
    col, avg, min, max, rank, row_number, count, sum,
    when, current_date, floor, months_between, datediff,
    to_date, concat, lit, upper, substring, coalesce,
    first, last, dense_rank, struct
)
from pyspark.sql.window import Window

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)

### Question 1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

Logic: The `pit_stops` dataset contains one row per pit stop event. Each row has `raceId`, `driverId`, and `milliseconds` (the duration of that stop). A driver can make multiple pit stops per race, so:

1. I group by `raceId` and `driverId` and compute the average pit stop duration per driver per race.
2. I group by `raceId` alone and compute the **minimum** (fastest) and **maximum** (slowest) individual pit stop across all drivers in each race.
3. I join the two results together so each row shows the driver's average alongside with the fastest and slowest results.

I use `milliseconds` (cast to integer) because it is the most precise numeric measure of pit stop duration.

In [0]:
df_pit_stops.head(10)

In [0]:
df_pit = df_pit_stops.withColumn("milliseconds", col("milliseconds").cast("int"))

# Average pit stop time per driver per race
df_avg_pit_per_driver = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(
        avg("milliseconds").alias("avg_pit_time_ms")
    )
)

display(df_avg_pit_per_driver)


In [0]:
# Fastest and slowest pit stop per race
df_race_pit_extremes = (
    df_pit
    .groupBy("raceId")
    .agg(
        min(col("milliseconds")) .alias("fastest_pit_stop_ms"),
        max(col("milliseconds")).alias("slowest_pit_stop_ms")
    )
)

# Combine them into one
df_q1 = (
    df_avg_pit_per_driver
    .join(df_race_pit_extremes, on="raceId", how="inner")
    .orderBy("raceId", "driverId")
)

display(df_q1)

### Code Explanation

1. **`.withColumn("milliseconds", col("milliseconds").cast("int"))`** — The CSV reader loads all columns as strings. This casts `milliseconds` to an integer so that `avg`, `min`, and `max` perform numeric operations.

2. **`.groupBy("raceId", "driverId").agg(avg("milliseconds").alias("avg_pit_time_ms"))`** — Groups pit stop records by race and driver, then computes the mean duration. `.alias()` renames the result column for clarification.

3. **`.groupBy("raceId").agg(min(...), max(...))`** — Groups at the race level to find the single fastest and slowest pit stop across all drivers and all stops in that race.

4. **`.join(..., on="raceId", how="inner")`** — Joins the per-driver averages.

5. **`.orderBy("raceId", "driverId")`** — Sorts for readability.

### Question 2：Rank order by finishing position the average time spent at the pit stop in each race.

Logic: This question asks: within each race, rank drivers by their finishing position and show their average pit stop time. The goal is to see whether drivers who finish higher tend to have faster pit stops.

1. Compute the average pit stop time per driver per race from `pit_stops`.
2. Get finishing positions from `results` (using `positionOrder`, which is always a clean integer — unlike `position` which can contain `\\N` for DNFs).
3. Join the two on `raceId` and `driverId`.
4. Order by `raceId` and `positionOrder` so that within each race, drivers are listed from 1st place downward.

In [0]:
# reuse df_pit from last question
df_avg_pit = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(avg("milliseconds").alias("avg_pit_time_ms"))
)

# Get finishing positions from results
df_finish = (
    df_results
    .select(
        col("raceId"),
        col("driverId"),
        col("positionOrder").cast("int").alias("finishing_position")
    )
)

# Join and order by finishing position within each race
df_q2 = (
    df_finish
    .join(df_avg_pit, on=["raceId", "driverId"], how="inner")
    .orderBy(col("raceId"), col("finishing_position"))
)

display(df_q2)

### Code Explanation

1. **`df_avg_pit`** — Groups `pit_stops` by `raceId` and `driverId` and computes the mean pit stop duration in milliseconds. This is the same logic as Q1.

2. **`df_finish`** — Selects `raceId`, `driverId`, and `positionOrder` from the `results` table. I use `positionOrder` rather than `position` because `positionOrder` is always a clean integer (1, 2, 3, ...) even for drivers who did not finish (DNF), while `position` can contain `\\N`. The `.cast("int")` converts the string to an integer for proper numeric sorting.

3. **`.join(..., on=["raceId", "driverId"], how="inner")`** — An inner join keeps only drivers who appear in both datasets (i.e., drivers who made at least one pit stop and have a race result). Drivers who never pitted (e.g., some DNFs on lap 1) are excluded.

4. **`.orderBy(col("raceId"), col("finishing_position"))`** — Sorts by race, then by finishing position within each race, so the output reads as a rank-ordered list.

### Extra Credit: Alternative Approach Using `rank()` Window Function

We can explicitly rank drivers by their average pit stop time within each race using the `rank()` window function, and then compare that pit stop ranking to their finishing position.

In [0]:
# Alternative: add an explicit pit stop rank column for comparison
pit_rank_window = Window.partitionBy("raceId").orderBy("avg_pit_time_ms")

df_q2_alt = (
    df_q2
    .withColumn("pit_stop_rank", rank().over(pit_rank_window))
    .orderBy(col("raceId"), col("finishing_position"))
)

display(df_q2_alt)

**Alternative explanation:** `Window.partitionBy("raceId").orderBy("avg_pit_time_ms")` creates a window partitioned by race and ordered by average pit time (ascending, so the fastest pit strategy gets rank 1). The `rank()` function assigns a rank within each race. This lets us directly compare `finishing_position` and `pit_stop_rank` — if they correlate, it suggests pit stop speed contributes to race outcomes.

### Question 3: Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

Logic: The `drivers` dataset has a `code` column representing a 3-letter abbreviation (e.g., "HAM" for Hamilton, "ALO" for Alonso). Some drivers have missing codes — stored as `\\N` in the CSV.

My approach to generate the missing codes:

1. Identify rows where `code` is `\\N` (missing).
2. For those drivers, generate a code using the first three letters of the driver's surname, converted to uppercase. This mirrors the convention used by the FIA for most modern drivers.
3. Use `when().otherwise()` to conditionally replace only the missing values, leaving existing valid codes untouched.
4. Verify there are no remaining missing codes.

In [0]:
# Check how many drivers have missing codes
print("Drivers with missing code (\\N):")
df_drivers.filter(col("code") == "\\N").select("driverId", "driverRef", "forename", "surname", "code").show(20, truncate=False)

In [0]:
# Fill missing codes with first 3 letters of surname
df_drivers_fixed = df_drivers.withColumn(
    "code",
    when(
        (col("code") == "\\N") | (col("code").isNull()),
        upper(substring(col("surname"), 1, 3))
    ).otherwise(col("code"))
)

display(df_drivers_fixed.select("driverId", "forename", "surname", "code"))

In [0]:
# Verify no missing codes remain
missing_count = df_drivers_fixed.filter((col("code") == "\\N") | (col("code").isNull())).count()
print(f"Remaining missing codes: {missing_count}")

### Code Explanation

1. **`.filter(col("code") == "\\N")`** — Identifies drivers whose `code` field is the placeholder string `\\N`, which represents missing data in this CSV format.

2. **`when((col("code") == "\\N") | (col("code").isNull()), ...).otherwise(col("code"))`** — A conditional expression: if the code is `\\N` or null, apply the replacement logic; otherwise, keep the existing code. This ensures we only modify rows that actually need fixing.

3. **`upper(substring(col("surname"), 1, 3))`** — `substring(col("surname"), 1, 3)` extracts the first 3 characters of the surname (1-indexed). `upper()` converts them to uppercase. For example, "Alonso" becomes "ALO", "Schumacher" becomes "SCH".

4. **The verification step** counts any remaining `\\N` or null codes to confirm all have been filled.
